In [1]:
import torch
import torch.nn as nn
import numpy as np

# Dataset
text = """machine learning models learn patterns from data.
sequence models process data step by step.
recurrent neural networks are designed for sequential tasks.
rnn models maintain hidden states across time steps.

long short term memory networks solve long dependency problems.
lstm uses gates to control information flow.
gru models simplify the lstm architecture.
sequence prediction is useful in many applications.

language modeling predicts the next word in a sentence.
speech recognition processes audio sequences.
time series forecasting predicts future values.
music generation creates new melodies.

generative models learn probability distributions.
they generate new samples similar to training data.
sequence generation is widely used in artificial intelligence.
deep learning improves sequence modeling performance.
"""


text = text.lower().split()
vocab = sorted(set(text))
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}


seq_length = 4
X = []
y = []

for i in range(len(text) - seq_length):
    seq = text[i:i+seq_length]
    target = text[i+seq_length]
    X.append([word2idx[w] for w in seq])
    y.append(word2idx[target])

X = torch.tensor(X)
y = torch.tensor(y)


class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

model = LSTMModel(len(vocab), 32, 64)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")


def generate(seed, length=10):
    model.eval()
    words = seed.split()

    for _ in range(length):
        inp = torch.tensor([[word2idx[w] for w in words[-seq_length:]]])
        with torch.no_grad():
            out = model(inp)
        next_word = idx2word[torch.argmax(out).item()]
        words.append(next_word)

    return " ".join(words)

# Test
print("\nGenerated Text:")
print(generate("machine learning models learn"))

Epoch 0, Loss: 4.487093925476074
Epoch 50, Loss: 0.0032365473452955484
Epoch 100, Loss: 0.0013677232200279832
Epoch 150, Loss: 0.000944021507166326

Generated Text:
machine learning models learn patterns from data. sequence models process data step by step.


In [2]:
import torch
import torch.nn as nn

# Reuse dataset
tokens = text
vocab = sorted(set(tokens))
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

seq_length = 4
X = []
y = []

for i in range(len(tokens) - seq_length):
    X.append([word2idx[w] for w in tokens[i:i+seq_length]])
    y.append(word2idx[tokens[i+seq_length]])

X = torch.tensor(X)
y = torch.tensor(y)


class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(1, 0, 2)
        out = self.transformer(x)
        out = self.fc(out[-1])
        return out

model = TransformerModel(len(vocab))

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


for epoch in range(200):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

# Generate sequence
def generate_transformer(seed, length=10):
    model.eval()
    words = seed.split()

    for _ in range(length):
        inp = torch.tensor([[word2idx[w] for w in words[-seq_length:]]])
        with torch.no_grad():
            out = model(inp)
        next_word = idx2word[torch.argmax(out).item()]
        words.append(next_word)

    return " ".join(words)

# Test
print("\nTransformer Generated Text:")
print(generate_transformer("deep learning improves sequence"))

/tmp/ipykernel_899/3290174257.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)


Epoch 0, Loss: 4.5531721115112305
Epoch 50, Loss: 0.002273691352456808
Epoch 100, Loss: 0.001236658776178956
Epoch 150, Loss: 0.0007397200097329915

Transformer Generated Text:
deep learning improves sequence modeling performance. samples similar to training data. sequence generation is
